# Stage 1: Experimental — Copy-wallet position signals

Select the **copy universe** (profitable, diversified, stable wallets)
from train-period metrics, then evaluate **generic aggregate-position
signals** of wallet archetypes (gamblers, whales, retail, oversellers,
scalpers, ...) on the candidate BUY trades:

- `pos_*`: aggregate open quantity the archetype holds on the token
- `val_*`: aggregate value-at-cost (USDC) held

Methodology (see the review in `position_signals.md`):

1. Forward copyable ROI is residualized against price (train-fitted,
   fixed to val/test) so the favorite/price effect (IC(price, roi) ~ +0.5)
   does not masquerade as wallet alpha.
2. Signals are the non-redundant pos/val x own/opp variants per archetype.
3. Selection = sign-consistent IC on train + validation **and** a
   bootstrap CI of the pooled train+val IC excluding 0.
4. Combination now uses a **train-fitted** rank normalizer before
   weighting, so validation/test rows are not normalized against their own
   split distribution.  Weights are fit on train, threshold on validation,
   and the strategy can be evaluated gross and net of taker cost.

The split is by market **end date** (`split_data(..., 'chronological')`)
so no contract spans the train/val/test boundary.  Library code lives in
`signal_lab/` (`signal_lib.py`, `signal_engines.py`, `stage1.py`), with
LLM-oriented workflow notes in `signal_lab/AGENTS.md`.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from IPython.display import display

from lib import (
    load_trades,
    split_data,
    compute_copyable_notional,
    compute_opening_metrics,
    DEFAULT_TAGS,
)
from polymarket_analysis.wallet_selection.volatility import compute_wallet_metrics
from signal_lab.signal_lib import (
    compute_event_ic,
    compute_event_ir,
    signal_quality_report,
    coincidence_rate,
    ic_correlation_matrix,
    fit_roi_residualizer,
    residualized_roi,
    cs_rank,
    fit_rank_transformer,
    apply_rank_transformer,
    compute_optimal_weights,
    apply_composite_score,
    evaluate_strategy,
)
from signal_lab.signal_engines import (
    PositionSignalEngine,
    compute_hold_time_metrics,
    archetype_sets,
    position_report,
)


## Parameters

In [ ]:
# Copy universe (train-period wallet metrics)
COPY_MIN_BUY_ROI = 0.02
COPY_MIN_BUCKETS = 20
COPY_MIN_MARKETS = 15
COPY_MIN_TRADE_COUNT = 100
COPY_MAX_DD_TO_PNL = 0.6
COPY_MIN_COPYABLE_ROI = 0.05

# Archetype universe
ARCH_MIN_TRADE_COUNT = 100

# Signal selection (on price-residualized ROI)
IC_ALPHA = 0.05
BOOT_ITER = 500
BOOT_SEED = 42
PRESENCE_MIN = 0.005
TAKER_COST_BPS = 0.0

# Non-redundant position-signal variants per archetype (pos/val x own/opp;
# 'total' = own+opp and avgc/uwl are algebraic composites of pos/val/price)
SIGNAL_KINDS = [('pos', 'own'), ('pos', 'opp'), ('val', 'own'), ('val', 'opp')]


## Load data

In [ ]:
df_full = load_trades()
df_full = compute_copyable_notional(df_full)
df_train, df_val, df_test = split_data(df_full, method='chronological')

print(f"Trades: full={len(df_full):,}  train={len(df_train):,}  "
      f"val={len(df_val):,}  test={len(df_test):,}")


## Compute wallet metrics on training data

In [ ]:
wallet_vol, _ = compute_wallet_metrics(df_train)

wallet_vol["copyable_pnl_factor"] = np.clip(
    wallet_vol["copyable_pnl"] / wallet_vol["total_pnl"].replace(0, np.nan),
    0, 1.0,
).fillna(0.0)
wallet_vol["copyable_roi"] = wallet_vol["average_roi"] * wallet_vol["copyable_pnl_factor"]

opening_metrics = compute_opening_metrics(df_train)
wallet_vol = wallet_vol.merge(opening_metrics, on="wallet", how="left")
for c in ["opening_roi", "opening_pnl", "opening_copyable_roi", "opening_copyable_pnl"]:
    wallet_vol[c] = wallet_vol[c].fillna(0.0)

print(f"Wallets with metrics: {len(wallet_vol)}")
wallet_vol[["wallet", "buy_roi", "sell_roi", "copyable_pnl",
           "copyable_roi", "num_buckets"]].head(10)


## Define the copy universe (candidate trades)

BUY trades by wallets that pass a quality + stability filter. These are the trades we could copy; **all evaluation** (selection ICs, quality, firing, PnL) is measured on candidate trades only. Signal *values* come from full-dataset archetype positions (the engine is built on `df_full`), but every IC and strategy metric below is computed on `c_train`/`c_val`/`c_test`.

In [ ]:
copy_mask = (
    (wallet_vol['buy_roi'] >= COPY_MIN_BUY_ROI)
    & (wallet_vol['num_buckets'] >= COPY_MIN_BUCKETS)
    & (wallet_vol['num_markets'] >= COPY_MIN_MARKETS)
    & (wallet_vol['trade_count'] >= COPY_MIN_TRADE_COUNT)
    & (wallet_vol['max_drawdown_to_pnl'].fillna(1.0) <= COPY_MAX_DD_TO_PNL)
    & (wallet_vol['copyable_roi'].fillna(0.0) >= COPY_MIN_COPYABLE_ROI)
)
copy_wallets = set(wallet_vol.loc[copy_mask, 'wallet'])
print(f"Copy universe: {len(copy_wallets)} wallets")

candidate_trades = df_full[
    df_full['wallet'].isin(copy_wallets) & (df_full['side'] == 'BUY')
].copy()
assert (candidate_trades['side'] == 'BUY').all(), "candidate trades must be BUY-only"

c_train, c_val, c_test = split_data(candidate_trades, method='chronological')
print(f"Candidate BUY trades: train={len(c_train):,}  val={len(c_val):,}  "
      f"test={len(c_test):,}")

conditions = set(candidate_trades['condition_id'].unique())
print(f"Candidate conditions: {len(conditions):,}")


## Archetype metrics and wallet sets

Per-wallet hold/flip-time metrics (train), then data-driven
archetype sets defined from quantiles over the active population:
whale, retail, gambler, overseller (deep/thin), consistent,
max_drawdown, both_sides, scalper, flipper.

In [ ]:
hold = compute_hold_time_metrics(df_train)
sets = archetype_sets(wallet_vol, hold, min_trade_count=ARCH_MIN_TRADE_COUNT)

rows = []
for name, sel in sets.items():
    w = wallet_vol[wallet_vol['wallet'].isin(sel['wallet'])]
    rows.append({'set': name, 'n_wallets': len(sel),
                 'total_pnl': w['total_pnl'].sum(),
                 'trades': int(w['trade_count'].sum()),
                 'copyable_roi': w['copyable_roi'].mean()})
display(pd.DataFrame(rows).round(4))


## Position signal engine

Builds once over the full trade frame: per-trade post-position
checkpoints with execution-order-aware value-at-cost.  Per-set A/B
tables are then a fast masked cumsum, and each candidate trade gets
the aggregate position / value-at-cost / entry-premium / underwater
state of the set on its token at that time.

In [ ]:
engine = PositionSignalEngine(df_full)


## Position signal sweep

For every archetype, attach the pos/val x own/opp signal variants
to the candidate trades and select train+val sign-consistent
signals whose pooled bootstrap CI excludes 0.  ICs are computed
against the **price-residualized ROI** (`roi_res`), fit on train
only.  This is the slow cell (11 archetypes x 3 candidate frames x
4 signal variants).

In [ ]:
# Fit the price residualizer on TRAIN only; apply fixed coefficients to val/test.
fit = fit_roi_residualizer(c_train['copyable_roi'], c_train['price'])
print(f'Residualizer (train): beta={fit["beta"]:+.4f}  '
      f'intercept={fit["intercept"]:+.4f}')
for lbl, df_c in [('train', c_train), ('val', c_val), ('test', c_test)]:
    df_c['roi_res'] = residualized_roi(df_c['copyable_roi'], df_c['price'], fit)
    ic_p = compute_event_ic(df_c['price'], df_c['roi_res'])
    print(f'  {lbl:5s}: IC(price, roi_res)={ic_p:+.4f}  '
          f'(~0 means the price effect is removed)')

all_rows, selected = [], []
for name, sel in sets.items():
    rep, sel_cols = position_report(
        engine, c_train, c_val, c_test,
        set(sel['wallet']), name,
        roi_col='roi_res', kinds=SIGNAL_KINDS,
        presence_min=PRESENCE_MIN, alpha=IC_ALPHA,
        n_boot=BOOT_ITER, seed=BOOT_SEED,
        conditions=conditions,
    )
    all_rows.append(rep)
    selected.extend(sel_cols)
    print(f'  {name}: n_wallets={len(sel):>4}  selected={sel_cols}')

report = pd.concat(all_rows, ignore_index=True)
print('\n=== Position signal residual-ICs (test = diagnostics) ===')
display(report.sort_values('|IC_train|', ascending=False).round(4))
print(f'\nSelected ({len(selected)}): {selected}')


## Signal quality framework

Following Grinold & Kahn: IC (Spearman rank correlation between
signal and the price-residualized forward ROI), IR (mean/std of
daily IC), hit rate (active events only), bootstrap CI.  Implemented in `signal_lib.py`.

In [ ]:
active_cols = [c for c in selected if c in c_val.columns
               and c_val[c].notna().sum() > 10]
print(f"Active signals: {len(active_cols)} / {len(selected)}")

if not active_cols:
    print('No active signals survived selection; diagnostics skipped')
else:
    quality = signal_quality_report(c_val, active_cols, roi_col='roi_res',
                                    dt_col='dt', n_bootstrap=1_000,
                                    bootstrap=True)
    display(quality.round(4))

    diag = []
    for c in active_cols:
        diag.append({
            'signal': c,
            'presence_train': float((c_train[c] > 0).mean()),
            'IC_train': compute_event_ic(c_train[c], c_train['roi_res']),
            'IC_val': compute_event_ic(c_val[c], c_val['roi_res']),
            'IC_test': compute_event_ic(c_test[c], c_test['roi_res']),
        })
    print('\nPer-split residual-IC diagnostics (test = out-of-sample):')
    display(pd.DataFrame(diag).round(4))


## Signal overlap analysis

How redundant are the selected signals? Coincidence rate (do they
fire together), IC correlation (are predictions redundant), and
conditional IC (unique contribution).

In [ ]:
if len(active_cols) >= 2:
    print("1. Coincidence rate (P(both fire | either fires)):")
    n = len(active_cols)
    coin = np.full((n, n), np.nan)
    for i, s1 in enumerate(active_cols):
        for j, s2 in enumerate(active_cols):
            coin[i, j] = 1.0 if i == j else coincidence_rate(c_val[s1], c_val[s2])
    display(pd.DataFrame(coin, index=active_cols, columns=active_cols).round(3))

    print("\n2. IC correlation (signal value correlation):")
    display(ic_correlation_matrix(c_val, active_cols).round(3))

    print("\n3. Conditional IC (unique contribution on neutral events):")
    for s in active_cols:
        other = [c for c in active_cols if c != s]
        neutral = np.ones(len(c_val), dtype=bool)
        for o in other:
            neutral &= (c_val[o].abs() < 0.01) | c_val[o].isna()
        if neutral.sum() < 20:
            continue
        ic_cond = compute_event_ic(c_val.loc[neutral, s], c_val.loc[neutral, 'roi_res'])
        ic_full = compute_event_ic(c_val[s], c_val['roi_res'])
        print(f"    {s:35s}: full_IC={ic_full:.4f}  "
              f"conditional_IC={ic_cond:.4f}  (n={neutral.sum()})")
else:
    print("Need at least 2 active signals for overlap analysis")


## Signal combination

Combine signals into a composite score.  Signals are **rank-
normalized** (`cs_rank`, maps to [-1, 1]) so dollar-scale
differences between whales and retail do not dominate.  The
normalizer is fit on **train only** and then applied unchanged to
validation/test, avoiding split-distribution leakage.  All
weight schemes are **direction-correct** (signed by IC) so
negative-IC signals contribute as short candidates.  Schemes:
equal magnitude, IC-weighted, and shrinkage Markowitz
(Grinold & Kahn Ch. 13) on the rank-transformed signals.

In [ ]:
if not active_cols:
    print('No active signals found')
else:
    # Fit signal normalization on TRAIN only, then apply unchanged to val/test.
    rank_cols = [f'rank_{c}' for c in active_cols]
    rank_fits = {c: fit_rank_transformer(c_train[c].fillna(0.0))
                 for c in active_cols}
    for c in active_cols:
        c_train[f'rank_{c}'] = apply_rank_transformer(c_train[c].fillna(0.0), rank_fits[c])
        c_val[f'rank_{c}'] = apply_rank_transformer(c_val[c].fillna(0.0), rank_fits[c])
        c_test[f'rank_{c}'] = apply_rank_transformer(c_test[c].fillna(0.0), rank_fits[c])

    ic_vals = {c: compute_event_ic(c_train[c], c_train['roi_res'])
               for c in active_cols}
    ic_signed = {c: (v if np.isfinite(v) else 0.0)
                 for c, v in ic_vals.items()}
    n = len(active_cols)
    w_equal = pd.Series({f'rank_{c}': np.sign(ic_signed[c]) / n
                         for c in active_cols})

    ic_sum = sum(abs(v) for v in ic_signed.values())
    w_ic = pd.Series({f'rank_{c}': (ic_signed[c] / ic_sum if ic_sum > 0
                                    else np.sign(ic_signed[c]) / n)
                     for c in active_cols})

    w_shrink = compute_optimal_weights(c_train, rank_cols, 'roi_res',
                                      shrinkage=0.5)

    schemes = {'equal': w_equal, 'ic_weighted': w_ic,
               'shrinkage_markowitz': w_shrink}
    for name, w in schemes.items():
        print(f'\n  {name}:')
        for c, wt in w.items():
            print(f'    {c:45s} = {wt:.4f}')
        for df_c in [c_train, c_val, c_test]:
            df_c[f'composite_{name}'] = apply_composite_score(df_c, rank_cols, w)

    comp_rows = []
    for name in schemes:
        cc = f'composite_{name}'
        comp_rows.append({
            'composite': cc,
            'IC': compute_event_ic(c_val[cc], c_val['roi_res']),
            'IR': compute_event_ir(c_val[cc], c_val['roi_res'], c_val['dt'],
                                   freq='D'),
        })
    print('\nComposite signal quality (validation, on residualized ROI):')
    display(pd.DataFrame(comp_rows).round(4))


## Strategy evaluation

When composite_score >= threshold, copy the BUY trade. Grid-search
the threshold on validation (max net copyable PnL); report gross
on the test set, always vs. the **copy-all-candidate-trades**
baseline.  Because thresholds near the bottom of the [-1, 1]
composite range fire almost everything, a **firing-band table**
(fire the top 25/50/75/90/99% of candidates) shows where selection
actually adds value vs. copying everything.  All counts are of
candidate trades.

In [ ]:
candidates = [c for c in ('composite_shrinkage_markowitz',
                          'composite_ic_weighted', 'composite_equal')
              if c in c_val.columns and c_val[c].notna().sum() >= 10]
if not candidates:
    print('No composite signal available; strategy evaluation skipped')
else:
    best_composite = candidates[0]
    print(f'Using: {best_composite}')

    thresholds = np.arange(-1.0, 1.01, 0.05)
    val_df = pd.DataFrame([evaluate_strategy(c_val, best_composite, t,
                                          cost_bps=TAKER_COST_BPS)
                           for t in thresholds])
    val_df['pnl_per_trade'] = (val_df['copyable_pnl_net']
                               / val_df['trades'].clip(lower=1))

    print('\nGrid search (validation): top 10 by copyable_pnl_net')
    display(val_df.sort_values('copyable_pnl_net', ascending=False)
            .head(10).round(2))

    cand = val_df[val_df['trades'] >= 20]
    best_row = (cand if not cand.empty else val_df)\
        .sort_values('copyable_pnl_net', ascending=False).iloc[0]
    best_threshold = float(best_row['threshold'])
    print(f'\nBest threshold: {best_threshold:.2f}  '
          f'(copyable_pnl_net=${best_row["copyable_pnl_net"]:,.0f}, '
          f'{best_row["trades"]} trades)')


In [ ]:
if 'best_threshold' not in globals():
    print('No strategy to evaluate (skipped above)')
else:
    test_result = evaluate_strategy(c_test, best_composite, best_threshold,
                                   cost_bps=TAKER_COST_BPS)
    all_result = evaluate_strategy(c_test, best_composite, -np.inf,
                                   cost_bps=TAKER_COST_BPS)

    print('Test set evaluation (candidate trades):')
    print(f'  Threshold: {best_threshold:.2f}')
    print(f'  Trades fired: {test_result["trades"]:,} / {len(c_test):,} '
          f'candidates ({test_result["firing_rate"]:.1%})')
    print(f'  Copyable PnL gross/net: ${test_result["copyable_pnl"]:,.0f} / '
          f'${test_result["copyable_pnl_net"]:,.0f}')
    print(f'  Copyable ROI gross/net: {test_result["copyable_roi"]:.4f} / '
          f'{test_result["copyable_roi_net"]:.4f}')
    print(f'  Cost paid: ${test_result["cost_paid"]:,.0f}')
    print(f'  Total PnL: ${test_result["total_pnl"]:,.0f}')
    print(f'  PnL per trade: '
          f'${test_result["copyable_pnl_net"] / max(test_result["trades"], 1):.2f}')
    print('\nvs. copying ALL candidate trades:')
    print(f'  Copyable PnL gross/net (all): ${all_result["copyable_pnl"]:,.0f} / '
          f'${all_result["copyable_pnl_net"]:,.0f}')
    print(f'  Copyable ROI gross/net (all): {all_result["copyable_roi"]:.4f} / '
          f'{all_result["copyable_roi_net"]:.4f}')

    print('\nFiring-band table (test set; fire top q of candidates by '
          'composite):')
    band_rows = []
    for q in [0.25, 0.50, 0.75, 0.90, 0.99]:
        thr = float(c_test[best_composite].quantile(1 - q))
        r = evaluate_strategy(c_test, best_composite, thr, cost_bps=TAKER_COST_BPS)
        band_rows.append({'fire': f'top {q:.0%}',
                          'threshold': thr,
                          'trades': r['trades'],
                          'firing_rate': r['firing_rate'],
                          'copyable_pnl': r['copyable_pnl'],
                          'copyable_pnl_net': r['copyable_pnl_net'],
                          'copyable_roi': r['copyable_roi'],
                          'copyable_roi_net': r['copyable_roi_net'],
                          'pnl_per_trade': (r['copyable_pnl_net']
                                            / max(r['trades'], 1))})
    band_rows.append({'fire': 'ALL candidates',
                      'threshold': -np.inf,
                      'trades': len(c_test),
                      'firing_rate': 1.0,
                      'copyable_pnl': all_result['copyable_pnl'],
                      'copyable_pnl_net': all_result['copyable_pnl_net'],
                      'copyable_roi': all_result['copyable_roi'],
                      'copyable_roi_net': all_result['copyable_roi_net'],
                      'pnl_per_trade': (all_result['copyable_pnl_net']
                                        / max(len(c_test), 1))})
    band_df = pd.DataFrame(band_rows)
    band_df['delta_vs_all'] = band_df['copyable_pnl_net'] - all_result['copyable_pnl_net']
    display(band_df.round(4))

    print('\n=== Strategy Summary (candidate trades) ===')
    for label, df_i in [('Train', c_train), ('Val', c_val), ('Test', c_test)]:
        r = evaluate_strategy(df_i, best_composite, best_threshold, cost_bps=TAKER_COST_BPS)
        ra = evaluate_strategy(df_i, best_composite, -np.inf, cost_bps=TAKER_COST_BPS)
        print(f'  {label:6s}: threshold={best_threshold:.2f}  '
              f'trades={r["trades"]:>5,}/{len(df_i):>6,}  '
              f'cpnl_net=${r["copyable_pnl_net"]:>8,.0f}  '
              f'croi_net={r["copyable_roi_net"]:.4f}  '
              f'(all: cpnl_net=${ra["copyable_pnl_net"]:>8,.0f}  '
              f'croi_net={ra["copyable_roi_net"]:.4f})')


## Framework self-check (synthetic signal)

Validate the IC machinery on a synthetic signal with a *known* Spearman correlation.  For a Gaussian copula with Pearson `rho`, the population Spearman is `6/pi * arcsin(rho/2)`; the framework must recover it.  (Full unit-level validation lives in `tests/test_signal_framework.py`.)

In [ ]:
rng = np.random.default_rng(123)
rho = 0.5
n = 50_000
z = rng.normal(size=n)
roi = rho * z + np.sqrt(1 - rho**2) * rng.normal(size=n)
expected = 6.0 / np.pi * np.arcsin(rho / 2)
ic = compute_event_ic(pd.Series(z), pd.Series(roi))
ir = compute_event_ir(
    pd.Series(z), pd.Series(roi),
    pd.Series(pd.date_range('2026-01-01', periods=n, freq='min', tz='UTC')),
)
print(f"known Spearman IC={expected:.4f}  recovered IC={ic:.4f}  daily IR={ir:.3f}")
assert abs(ic - expected) < 0.01, 'IC does not recover the known synthetic signal'
print('Framework self-check OK')
